In [2]:
import os
import torch 
import torch.nn as nn
from torch.utils.data import DataLoader
import torch.optim as op
from torch.amp import GradScaler
import optuna
import pickle
from tqdm import tqdm
# custom
import custom.models as mod
import custom.tools as tl

if torch.cuda.is_available():
    device = torch.device('cuda')
    torch.backends.cudnn.benchmark = True
else: device = torch.device('cpu')
print(f'{device=}')

device=device(type='cuda')


In [ ]:
## Useful parameters:
batch_size = 256
num_epochs = 50
win_len = 100
win_stride = 30
num_workers = 10
persistent_workers = True if num_workers>0 else False

main_dir = "/mnt/eph/multi_feat_model_stuff/"
h5file= "L1_70_12_6_6.h5"
h5_path = f"/mnt/eph/data/{h5file}"
train = tl.h5set(h5_path, win_len=win_len, win_stride=win_stride, name='A')
val = tl.h5set(h5_path, win_len=win_len, win_stride=win_stride, name='B')

train_loader = DataLoader(train, batch_size=batch_size, num_workers=os.cpu_count())
validation = DataLoader(data_val, batch_size=batch_size, num_workers=os.cpu_count())

## DB
optuna_url = f"sqlite:///{main_dir}optuna_study.db"
artifact_path = f"{main_dir}artifacts/"
artifact_store = optuna.artifacts.FileSystemArtifactStore(base_path=artifact_path)
trials_dir = f"{main_dir}trials/"
artifact_store = optuna.artifacts.FileSystemArtifactStore(base_path='/mnt/eph/artifacts')

def objective(trial:optuna.Trial):
    loss_func = nn.MSELoss()
    
    # Trial choices
    optim_name = trial.suggest_categorical("optimizer", ['Adam', 'SGD', 'RMSprop', 'NAdam'])
    lr = trial.suggest_float("learning_rate", 1e-5, 1e-1, log=True)
    decay = trial.suggest_categorical('decay', [0, 1e-6, 1e-4, 1e-5])
    clip = trial.suggest_float('clip', .2, 5)
    
    # Training phase
    # activation = getattr(nn, activation)
    model = mod.LSTMAEUni().to(device)
    if optim_name == 'SGD':
        momentum = trial.suggest_float("momentum", 0.3, 0.9, step=0.1)
        nesterov = trial.suggest_categorical('nesterov', [True, False])
        optim = getattr(op, optim_name)(model.parameters(), lr=lr, momentum=momentum, nesterov=nesterov)
    elif optim_name == 'RMSprop':
        momentum = trial.suggest_float("momentum", 0.3, 0.9, step=0.1)
        optim = getattr(op, optim_name)(model.parameters(), lr=lr, momentum=momentum)
    else: optim = getattr(op, optim_name)(model.parameters(), lr=lr)
    
    losses = {'train':[], 'val':[]}
    scaler = GradScaler()
    for epoch in range(1, num_epochs + 1):
        train_loss = tl.train_epoch(model, device, train_loader, loss_func, optim, scaler, clip, False)
        val_loss = tl.val_epoch(model, device, test_loader, loss_func, False)
        print(f'TRAIN - EPOCH {epoch}/{num_epochs} - loss: {train_loss} - test loss: {val_loss}')
        trial.report(val_loss, epoch)
        if trial.should_prune(): raise optuna.TrialPruned()
        losses['train'].append(train_loss)
        losses['val'].append(val_loss)

    torch.save({
        'model_state_dict': model.state_dict(),
        'optim_state_dict': optimizer.state_dict(),
        'scaler_state_dict': scaler.state_dict()
    }, 'model_opt_scaler.pt')

    with open('losses.pickle', 'rb') as fout: pickle.dump(losses, fout, protocol=-1)

    mod_opt_scaler_id = optuna.artifacts.upload_artifact(artifact_store=artifact_store, file_path='model_opt_scaler.pt', study_or_trial=trial.study)
    losses_id = optuna.artifacts.upload_artifact(artifact_store=artifact_store, file_path='losses.pickle', study_or_trial=trial.study)

    trial.set_user_attr('model_optim_id', art_id)
    trial.set_user_attr('dict_loss_id', dict_id)    
    return val_loss


optuna.logging.get_logger("optuna").addHandler(logging.StreamHandler(sys.stdout))
# to change model, just change study_name, don't touch storage
study = optuna.create_study(direction='minimize', study_name='Eric', storage=f"sqlite:///CanOTuna.db", load_if_exists=True)
study.optimize(objective, n_trials=30)